In [ ]:
# importing libraries for sql analysis

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine

In [2]:
#Connect to postgresql

In [3]:
username = "postgres"
password = "password"
host = "localhost"
port = "5432"
database = "technova_finance"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

with engine.connect() as conn:
    print("Connected Successfully!")

Connected Successfully!


In [4]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
"""

pd.read_sql(query,engine)

,table_name
0,dim_customer
1,fact_sales
2,dim_products
3,dim_salesperson
4,dim_date
5,dim_geography


## KPI Analysis

In [5]:
pd.options.display.float_format = '{:,.2f}'.format

#### What is the total revenue?

In [6]:
def format_dollars(value):
    if value >= 1_000_000:
        return f"{value/1_000_000:.2f}M"
    elif value >=1000:
        return f"{value/1000:.2f}K"
    else:
        return round(value,2)

In [7]:
query = """
SELECT SUM(net_sales) AS total_revenue FROM fact_sales
"""

total_revenue = pd.read_sql(query,engine)
value = total_revenue.loc[0,'total_revenue']

print(format_dollars(value))

1813.86M


#### What is the total profit?

In [8]:
query = """
SELECT SUM(profit) AS total_profit
FROM fact_sales
WHERE net_sales > 0
"""

total_profit = pd.read_sql(query,engine)
print(format_dollars(total_profit.loc[0,'total_profit']))

503.83M


#### What is the total number of orders?

In [9]:
query = """
SELECT DISTINCT(COUNT(order_id)) AS total_order_count
FROM fact_sales
WHERE net_sales > 0
"""

total_orders = pd.read_sql(query,engine)
total_orders

,total_order_count
0,75000


#### What is the total quantity sold?

In [10]:
query = """
SELECT SUM(quantity) AS total_quantity
FROM fact_sales
WHERE net_sales > 0
"""

total_quantity = pd.read_sql(query,engine)
total_quantity

,total_quantity
0,"1,913,241.00"


#### What is the average order value (AOV)?

In [11]:
query = """
SELECT (SUM(net_sales) / COUNT(DISTINCT order_id)) AS AOV
FROM fact_sales
WHERE net_sales > 0
"""

aov = pd.read_sql(query,engine)
aov = aov.loc[0,'aov'].round(2)
print(f"Average Order Value: {format_dollars(aov)}")

Average Order Value: 24.18K


### What is the average selling price?

In [12]:
query = """
SELECT SUM(net_sales)/SUM(quantity) AS average_selling_price
FROM fact_sales
WHERE net_sales > 0
"""

avg_selling_price = pd.read_sql(query,engine)
avg_selling_price = avg_selling_price.loc[0,'average_selling_price'].round(2)
print(f"Average Selling Price: {format_dollars(avg_selling_price)}$")

Average Selling Price: 948.06$


#### What is the overall profit margin?

In [13]:
query = """
SELECT (SUM(profit) / SUM(net_sales))*100 AS overall_profit_margin
FROM fact_sales
WHERE net_sales > 0
"""
overall_profit_margin = pd.read_sql(query,engine)
overall_profit_margin = overall_profit_margin.loc[0,'overall_profit_margin'].round(2)
print(f"Overall Profit Margin: {overall_profit_margin} $")

Overall Profit Margin: 27.78 $


#### Profit Margin for each year

In [14]:
query = """
SELECT TO_CHAR(order_date::date,'YYYY') AS year,
       (SUM(profit) / SUM(net_sales))*100 AS yearly_profit_margin
FROM fact_sales
GROUP BY 1
ORDER BY 2 DESC
"""

pd.read_sql(query,engine)

,year,yearly_profit_margin
0,2022,27.83
1,2023,27.83
2,2025,27.73
3,2024,27.73


Insights:

- total revenue = 1813.86M
- total profit = 503.83M
- total orders = 75000
- total quantity = 1,913,241.00
- AOV = 24.18K
- Average Selling Price: 948.06
- Overall Profit Margin: 27.78%


## Revenue Analysis

#### Is revenue actually declining year over year?

In [15]:
query = """
SELECT TO_CHAR(order_date::date,'YYYY') AS year,
       SUM(net_sales) as revenue  
FROM fact_sales
GROUP BY 1
ORDER BY 2 DESC
"""

highest_revenue = pd.read_sql(query,engine)
highest_revenue['revenue'] = highest_revenue['revenue'].apply(format_dollars)
display(highest_revenue)

,year,revenue
0,2023,458.65M
1,2024,454.75M
2,2022,454.73M
3,2025,445.73M


#### What is the YoY growth rate for revenue?

In [16]:
query = """
WITH yearly_sales AS
(
SELECT TO_CHAR(order_date::date,'YYYY') AS year,
       SUM(net_sales) as revenue
FROM fact_sales
GROUP BY 1
),
prev_year_sales AS
(
SELECT year,
       revenue,
       LAG(revenue) OVER (ORDER BY year) AS prev_year_revenue
FROM yearly_sales
)
SELECT year,
       (revenue - prev_year_revenue)*100 /NULLIF(prev_year_revenue,0) AS yoy_revenue_growth_percentage
       FROM prev_year_sales
"""

pd.read_sql(query,engine)


,year,yoy_revenue_growth_percentage
0,2022,NaN
1,2023,0.86
2,2024,-0.85
3,2025,-1.98


#### Which months show the biggest decline?

In [53]:
query = """
WITH monthly_sales  AS
(
SELECT d.year,
       d.month,
       d.month_name,
       SUM(f.net_sales) AS revenue
FROM fact_sales f
JOIN dim_date d ON f.order_date = d.date
GROUP BY d.year,d.month,d.month_name
),
monthly_growth AS
(
SELECT year,
       month,
       month_name,
       revenue,
       LAG(revenue) OVER (PARTITION BY month ORDER BY year) AS previous_year_revenue
FROM monthly_sales
)

SELECT year,
       month_name,
       ROUND(revenue::numeric,2) AS revenue,
       ROUND(previous_year_revenue::numeric,2) AS previous_year_revenue,
       ((revenue - previous_year_revenue)/previous_year_revenue)*100 AS yoy_growth_rate
FROM monthly_growth
WHERE year IN (2025)
"""

df = pd.read_sql(query,engine)
df

,year,month_name,revenue,previous_year_revenue,yoy_growth_rate
0,2025,January,"39,271,037.80","37,748,901.90",4.03
1,2025,February,"31,946,926.25","38,773,220.20",-17.61
2,2025,March,"36,791,968.85","38,216,383.45",-3.73
3,2025,April,"38,271,850.95","37,395,684.20",2.34
4,2025,May,"38,647,284.90","37,881,767.80",2.02
5,2025,June,"37,326,900.60","38,420,832.60",-2.85
6,2025,July,"37,331,551.00","40,447,482.70",-7.70
7,2025,August,"37,292,267.30","39,978,350.55",-6.72
8,2025,September,"37,661,472.60","34,894,630.70",7.93
9,2025,October,"37,028,769.00","39,766,586.25",-6.88


#### Which regions lost the most revenue?

In [18]:
query = """
WITH regional_sales AS
(
SELECT d.year,
       g.region,
       SUM(f.net_sales) AS revenue
FROM fact_sales f
JOIN dim_date d ON f.order_date = d.date
JOIN dim_geography g ON f.geo_id = g.geo_id
GROUP BY d.year,g.region
),
regional_sales_growth AS
(
SELECT year,
       region,
       revenue,
       LAG(revenue)OVER(PARTITION BY region ORDER BY year) AS previous_year_revenue
FROM regional_sales
)

SELECT year,
       region,
       ROUND(revenue::numeric,2) AS revenue,
       ROUND(previous_year_revenue::numeric,2) AS previous_year_revenue,
       ((revenue - previous_year_revenue)/NULLIF(previous_year_revenue,0))*100 AS yoy_growth_rate
FROM regional_sales_growth
"""

lost_revenue_region = pd.read_sql(query,engine)
lost_revenue_region

,year,region,revenue,previous_year_revenue,yoy_growth_rate
0,2022,Asia-Pacific,"70,321,335.00",NaN,NaN
1,2023,Asia-Pacific,"69,070,406.80","70,321,335.00",-1.78
2,2024,Asia-Pacific,"69,450,203.55","69,070,406.80",0.55
3,2025,Asia-Pacific,"67,946,785.30","69,450,203.55",-2.16
4,2022,Europe,"86,184,002.85",NaN,NaN
5,2023,Europe,"83,632,502.60","86,184,002.85",-2.96
6,2024,Europe,"84,356,923.10","83,632,502.60",0.87
7,2025,Europe,"79,859,835.35","84,356,923.10",-5.33
8,2022,Middle East,"81,624,109.20",NaN,NaN
9,2023,Middle East,"85,010,557.20","81,624,109.20",4.15


#### Which countries contributed most to the decline?

In [19]:
query = """
WITH country_sales AS
(
SELECT d.year,
       g.country,
       SUM(f.net_sales) AS revenue
FROM fact_sales f
JOIN dim_date d ON f.order_date = d.date
JOIN dim_geography g ON f.geo_id = g.geo_id
GROUP BY d.year,g.country
),
country_sales_growth AS
(
SELECT year,
       country,
       revenue,
       LAG(revenue)OVER(PARTITION BY country ORDER BY year) AS previous_year_revenue
FROM country_sales
)

SELECT year,
       country,
       ROUND(revenue::numeric,2) AS revenue,
       ROUND(previous_year_revenue::numeric,2) AS previous_year_revenue ,
       ((revenue - previous_year_revenue)/NULLIF(previous_year_revenue,0))*100 AS yoy_growth_rate
FROM country_sales_growth
"""


country_sales_declined= pd.read_sql(query,engine)
country_sales_declined

,year,country,revenue,previous_year_revenue,yoy_growth_rate
0,2022,Argentina,"44,080,921.20",NaN,NaN
1,2023,Argentina,"44,637,617.80","44,080,921.20",1.26
2,2024,Argentina,"45,104,581.30","44,637,617.80",1.05
3,2025,Argentina,"46,067,110.95","45,104,581.30",2.13
4,2022,Australia,"15,002,729.15",NaN,NaN
5,2023,Australia,"16,290,676.60","15,002,729.15",8.58
6,2024,Australia,"14,096,445.90","16,290,676.60",-13.47
7,2025,Australia,"14,535,246.30","14,096,445.90",3.11
8,2022,Brazil,"58,111,236.50",NaN,NaN
9,2023,Brazil,"55,960,756.70","58,111,236.50",-3.70


#### Which customer segments revenue declined?

In [20]:
query = """
WITH customer_segment_sales AS
(
SELECT d.year,
       c.segment,
       SUM(f.net_sales) AS revenue
FROM fact_sales f
JOIN dim_date d ON f.order_date = d.date
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY d.year,c.segment
),
customer_segment_sales_growth AS
(
SELECT year,
       segment,
       revenue,
       LAG(revenue)OVER(PARTITION BY segment ORDER BY year) AS previous_year_revenue
FROM customer_segment_sales
)

SELECT year,
       segment,
       ROUND(revenue::numeric,2) AS revenue,
       ROUND(previous_year_revenue::numeric,2) AS previous_year_revenue,
       ((revenue - previous_year_revenue)/NULLIF(previous_year_revenue,0))*100 AS yoy_growth_rate
FROM customer_segment_sales_growth

"""

customer_segment = pd.read_sql(query,engine)
customer_segment

,year,segment,revenue,previous_year_revenue,yoy_growth_rate
0,2022,Enterprise,"115,809,779.25",NaN,NaN
1,2023,Enterprise,"117,440,424.80","115,809,779.25",1.41
2,2024,Enterprise,"115,392,547.35","117,440,424.80",-1.74
3,2025,Enterprise,"113,553,060.30","115,392,547.35",-1.59
4,2022,Government,"108,125,875.50",NaN,NaN
5,2023,Government,"110,278,986.20","108,125,875.50",1.99
6,2024,Government,"106,383,452.55","110,278,986.20",-3.53
7,2025,Government,"108,755,707.05","106,383,452.55",2.23
8,2022,Retail,"118,437,325.95",NaN,NaN
9,2023,Retail,"120,056,585.70","118,437,325.95",1.37


#### Which industries revenue declined?

In [21]:
query = """
WITH customer_industry_sales AS
(
SELECT d.year,
       c.industry,
       SUM(f.net_sales) AS revenue
FROM fact_sales f
JOIN dim_date d ON f.order_date = d.date
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY d.year,c.industry
),
customer_industry_sales_growth AS
(
SELECT year,
       industry,
       revenue,
       LAG(revenue)OVER(PARTITION BY industry ORDER BY year) AS previous_year_revenue
FROM customer_industry_sales
)

SELECT year,
       industry,
       ROUND(revenue::numeric,2) AS revenue,
       ROUND(previous_year_revenue::numeric,2) AS previous_year_revenue,
       ((revenue - previous_year_revenue)/NULLIF(previous_year_revenue,0))*100 AS yoy_growth_rate
FROM customer_industry_sales_growth

"""

industry_declined = pd.read_sql(query,engine)
(industry_declined)

,year,industry,revenue,previous_year_revenue,yoy_growth_rate
0,2022,Finance,"109,392,486.85",NaN,NaN
1,2023,Finance,"112,732,575.85","109,392,486.85",3.05
2,2024,Finance,"109,638,649.15","112,732,575.85",-2.74
3,2025,Finance,"107,998,914.45","109,638,649.15",-1.50
4,2022,Healthcare,"116,292,748.85",NaN,NaN
5,2023,Healthcare,"115,665,404.15","116,292,748.85",-0.54
6,2024,Healthcare,"114,203,078.30","115,665,404.15",-1.26
7,2025,Healthcare,"111,178,261.00","114,203,078.30",-2.65
8,2022,IT,"118,536,741.30",NaN,NaN
9,2023,IT,"118,702,683.40","118,536,741.30",0.14


#### Which product categories revenue declined?

In [22]:
query = """
WITH product_category_sales AS
(
SELECT d.year,
       p.category,
       SUM(f.net_sales) AS revenue
FROM fact_sales f
JOIN dim_date d ON f.order_date = d.date
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY d.year,p.category
),
product_category_sales_growth AS
(
SELECT year,
       category,
       revenue,
       LAG(revenue)OVER(PARTITION BY category ORDER BY year) AS previous_year_revenue
FROM product_category_sales
)

SELECT year,
       category,
       ROUND(revenue::numeric,2) AS revenue,
       ROUND(previous_year_revenue::numeric,2) AS previous_year_revenue,
       ((revenue - previous_year_revenue)/NULLIF(previous_year_revenue,0))*100 AS yoy_growth_rate
FROM product_category_sales_growth

"""

pd.read_sql(query,engine)

,year,category,revenue,previous_year_revenue,yoy_growth_rate
0,2022,Laptop,"80,335,957.60",NaN,NaN
1,2023,Laptop,"79,653,470.15","80,335,957.60",-0.85
2,2024,Laptop,"77,979,767.75","79,653,470.15",-2.10
3,2025,Laptop,"79,013,091.05","77,979,767.75",1.33
4,2022,Monitor,"84,322,938.45",NaN,NaN
5,2023,Monitor,"86,542,432.75","84,322,938.45",2.63
6,2024,Monitor,"89,750,323.75","86,542,432.75",3.71
7,2025,Monitor,"87,598,374.80","89,750,323.75",-2.40
8,2022,Networking,"88,624,638.50",NaN,NaN
9,2023,Networking,"90,824,838.00","88,624,638.50",2.48


# Revenue Insights

- **2025 revenue declined across the business**, with weakness observed across most regions, customer segments, industries, and product categories.
- Revenue decline was **concentrated in specific periods**, particularly **February** and **July–October**, indicating seasonal or
operational challenges rather than a continuous downward trend.
- **All regions recorded negative YoY growth in 2025**, with **Europe** posting the largest decline, while **North America** remained 
the highest revenue contributor.
- **Country performance was mixed**: **Argentina** delivered consistent growth, whereas **Germany, Canada, France, and the UK** 
experienced sustained revenue declines.
- **Retail, Enterprise, and SMB** customer segments declined in 2025, while **Government** was the **only segment to achieve positive growth**.
- **Healthcare and Retail** were the weakest-performing industries, whereas **IT** was the **only industry to record positive growth** in 2025.
- **Networking, Storage, and Monitor** were the primary product categories driving the revenue decline, while **Software and Laptop** 
returned to positive growth.
- Strong recoveries in **September, November, and December** indicate opportunities to replicate successful strategies during weaker periods.
- The findings highlight the need to **focus on underperforming months, regions, customer segments, industries, and product categories** 
to improve future revenue performance.

## Order Analysis

#### Did order volume decrease?

In [23]:
query = """
WITH yearly_orders AS 
(
SELECT d.year,
       COUNT(DISTINCT s.order_id) AS order_count
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
GROUP BY 1
),
order_growth AS
(
SELECT year,
       order_count,
       LAG(order_count::numeric)OVER(ORDER BY year) AS previous_year_orders
FROM yearly_orders
)
SELECT year,
       order_count,
       previous_year_orders,
       ROUND(
        ((order_count - previous_year_orders)
        / NULLIF(previous_year_orders, 0)) * 100,
        2
    ) AS yoy_growth_rate
FROM order_growth
"""

pd.read_sql(query,engine)

,year,order_count,previous_year_orders,yoy_growth_rate
0,2022,18826,NaN,NaN
1,2023,18691,"18,826.00",-0.72
2,2024,18936,"18,691.00",1.31
3,2025,18547,"18,936.00",-2.05


#### Did Average Order Value decrease?

In [57]:
query = """
WITH yearly_order_value AS
(
SELECT d.year,
       (SUM(f.net_sales) / COUNT(DISTINCT f.order_id)) AS aov
FROM fact_sales f
JOIN dim_date d ON f.order_date = d.date
GROUP BY d.year
),
yearly_order_growth AS
(
SELECT year,
       aov,
       LAG(aov)OVER(ORDER BY year) AS previous_year_aov
FROM yearly_order_value
)
SELECT year,
       aov,
       previous_year_aov,
       ((aov-previous_year_aov)/previous_year_aov)*100 AS yoy_growth_rate
FROM yearly_order_growth
"""

average_order_value = pd.read_sql(query,engine)
average_order_value

,year,aov,previous_year_aov,yoy_growth_rate
0,2022,"24,154.21",NaN,NaN
1,2023,"24,538.81","24,154.21",1.59
2,2024,"24,014.92","24,538.81",-2.13
3,2025,"24,032.70","24,014.92",0.07


#### Which products lost the most orders?

In [25]:
query = """
WITH products_order_count AS
(
SELECT d.year,
       p.product_name,
       COUNT(DISTINCT s.order_id) AS order_count
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
JOIN dim_products p ON s.product_id = p.product_id
GROUP BY d.year, p.product_name
),
product_order_growth AS
(
SELECT year,
       product_name,
       order_count,
       LAG(order_count::numeric)OVER(PARTITION BY product_name ORDER BY year) AS previous_year_count
FROM products_order_count
)
SELECT year,
       product_name,
       order_count,
       previous_year_count,
       (order_count - previous_year_count) AS order_loss,
       ((order_count - previous_year_count)/NULLIF(previous_year_count,0))*100 AS yoy_growth_rate
FROM product_order_growth
WHERE year = 2025
ORDER BY order_loss
LIMIT 20
"""

pd.read_sql(query,engine)

,year,product_name,order_count,previous_year_count,order_loss,yoy_growth_rate
0,2025,Network Hub 6,42,79.00,-37.00,-46.84
1,2025,SSD 512GB 6,47,78.00,-31.00,-39.74
2,2025,WiFi Router AX6000 9,46,75.00,-29.00,-38.67
3,2025,Network Hub 3,58,87.00,-29.00,-33.33
4,2025,Portable SSD 6,55,84.00,-29.00,-34.52
5,2025,Access Point Pro 6,46,74.00,-28.00,-37.84
6,2025,Crystal Display 4,52,80.00,-28.00,-35.00
7,2025,SSD 1TB 3,55,81.00,-26.00,-32.10
8,2025,Network Hub 12,57,83.00,-26.00,-31.33
9,2025,WiFi Router AX6000 5,57,83.00,-26.00,-31.33


#### Which product categories lost the most orders?

In [26]:
query = """
WITH products_category_count AS
(
SELECT d.year,
       p.category,
       COUNT(DISTINCT s.order_id) AS order_count
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
JOIN dim_products p ON s.product_id = p.product_id
GROUP BY d.year, p.category
),
product_category_growth AS
(
SELECT year,
       category,
       order_count,
       LAG(order_count::numeric)OVER(PARTITION BY category ORDER BY year) AS previous_year_count
FROM products_category_count
)
SELECT year,
       category,
       order_count,
       previous_year_count,
       (order_count - previous_year_count) AS order_loss,
       ((order_count - previous_year_count)/NULLIF(previous_year_count,0))*100 AS yoy_growth_rate
FROM product_category_growth
WHERE year = 2025
ORDER BY order_loss

"""

pd.read_sql(query,engine)

,year,category,order_count,previous_year_count,order_loss,yoy_growth_rate
0,2025,Networking,3530,"3,798.00",-268.00,-7.06
1,2025,Laptop,3190,"3,271.00",-81.00,-2.48
2,2025,Monitor,3640,"3,696.00",-56.00,-1.52
3,2025,Storage,3883,"3,917.00",-34.00,-0.87
4,2025,Software,4304,"4,254.00",50.00,1.18


#### Which regions lost the most orders?

In [27]:
query = """
WITH products_region_count AS
(
SELECT d.year,
       g.region,
       COUNT(DISTINCT s.order_id) AS order_count
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
JOIN dim_geography g ON s.geo_id = g.geo_id
GROUP BY d.year, g.region
),
product_region_growth AS
(
SELECT year,
       region,
       order_count,
       LAG(order_count::numeric)OVER(PARTITION BY region ORDER BY year) AS previous_year_count
FROM products_region_count
)
SELECT year,
       region,
       order_count,
       previous_year_count,
       (order_count - previous_year_count) AS order_loss,
       ((order_count - previous_year_count)/NULLIF(previous_year_count,0))*100 AS yoy_growth_rate
FROM product_region_growth
WHERE year = 2025
ORDER BY order_loss

"""

pd.read_sql(query,engine)

,year,region,order_count,previous_year_count,order_loss,yoy_growth_rate
0,2025,Europe,3342,"3,478.00",-136.00,-3.91
1,2025,South America,4217,"4,314.00",-97.00,-2.25
2,2025,North America,4811,"4,878.00",-67.00,-1.37
3,2025,Asia-Pacific,2781,"2,836.00",-55.00,-1.94
4,2025,Middle East,3396,"3,430.00",-34.00,-0.99


#### Which countries lost the most orders?

In [28]:
query = """
WITH products_country_count AS
(
SELECT d.year,
       g.country,
       COUNT(DISTINCT s.order_id) AS order_count
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
JOIN dim_geography g ON s.geo_id = g.geo_id
GROUP BY d.year, g.country
),
product_country_growth AS
(
SELECT year,
       country,
       order_count,
       LAG(order_count::numeric)OVER(PARTITION BY country ORDER BY year) AS previous_year_count
FROM products_country_count
)
SELECT year,
       country,
       order_count,
       previous_year_count,
       (order_count - previous_year_count) AS order_loss,
       ((order_count - previous_year_count)/NULLIF(previous_year_count,0))*100 AS yoy_growth_rate
FROM product_country_growth
WHERE year = 2025
ORDER BY order_loss

"""

pd.read_sql(query,engine)

,year,country,order_count,previous_year_count,order_loss,yoy_growth_rate
0,2025,Brazil,2312,"2,436.00",-124.00,-5.09
1,2025,Canada,1659,"1,747.00",-88.00,-5.04
2,2025,UAE,1681,"1,768.00",-87.00,-4.92
3,2025,France,1379,"1,457.00",-78.00,-5.35
4,2025,India,1372,"1,423.00",-51.00,-3.58
5,2025,Germany,885,918.00,-33.00,-3.59
6,2025,UK,1078,"1,103.00",-25.00,-2.27
7,2025,Japan,794,798.00,-4.00,-0.50
8,2025,Australia,615,615.00,0.00,0.00
9,2025,USA,3152,"3,131.00",21.00,0.67


#### Which customer segments placed fewer orders?

In [29]:
query = """
WITH products_segment_count AS
(
SELECT d.year,
       c.segment,
       COUNT(DISTINCT s.order_id) AS order_count
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
JOIN dim_customer c ON s.customer_id = c.customer_id
GROUP BY d.year, c.segment
),
product_segment_growth AS
(
SELECT year,
       segment,
       order_count,
       LAG(order_count::numeric)OVER(PARTITION BY segment ORDER BY year) AS previous_year_count
FROM products_segment_count
)
SELECT year,
       segment,
       order_count,
       previous_year_count,
       (order_count - previous_year_count) AS order_loss,
       ((order_count - previous_year_count)/NULLIF(previous_year_count,0))*100 AS yoy_growth_rate
FROM product_segment_growth
WHERE year = 2025
ORDER BY order_loss

"""

pd.read_sql(query,engine)

,year,segment,order_count,previous_year_count,order_loss,yoy_growth_rate
0,2025,SMB,4446,"4,718.00",-272.00,-5.77
1,2025,Retail,4811,"4,950.00",-139.00,-2.81
2,2025,Government,4464,"4,506.00",-42.00,-0.93
3,2025,Enterprise,4826,"4,762.00",64.00,1.34


#### Which industries reduced their purchases?

In [30]:
query = """
WITH products_industry_count AS
(
SELECT d.year,
       c.industry,
       COUNT(DISTINCT s.order_id) AS order_count
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
JOIN dim_customer c ON s.customer_id = c.customer_id
GROUP BY d.year, c.industry
),
product_industry_growth AS
(
SELECT year,
       industry,
       order_count,
       LAG(order_count::numeric)OVER(PARTITION BY industry ORDER BY year) AS previous_year_count
FROM products_industry_count
)
SELECT year,
       industry,
       order_count,
       previous_year_count,
       (order_count - previous_year_count) AS order_loss,
       ((order_count - previous_year_count)/NULLIF(previous_year_count,0))*100 AS yoy_growth_rate
FROM product_industry_growth
WHERE year = 2025
ORDER BY order_loss

"""

pd.read_sql(query,engine)

,year,industry,order_count,previous_year_count,order_loss,yoy_growth_rate
0,2025,Finance,4491,"4,637.00",-146.00,-3.15
1,2025,Healthcare,4658,"4,793.00",-135.00,-2.82
2,2025,Retail,4671,"4,751.00",-80.00,-1.68
3,2025,IT,4727,"4,755.00",-28.00,-0.59


#### Which months had the largest decline?

In [31]:
query = """
WITH monthly_order_count AS
(
SELECT d.year,
       d.month,
       d.month_name,
       COUNT(DISTINCT s.order_id) AS order_count
FROM fact_sales s 
JOIN dim_date d ON s.order_date = d.date
GROUP BY d.year,d.month,d.month_name
),
monthly_order_growth AS
(
SELECT year,
       month,
       month_name,
       order_count,
       LAG(order_count::numeric)OVER(PARTITION BY month ORDER BY year) AS previous_order_count
FROM monthly_order_count
)
SELECT year,
       month_name,
       order_count,
       previous_order_count,
       (order_count - previous_order_count) AS order_loss_gain,
       ((order_count - previous_order_count)/NULLIF(previous_order_count,0))*100 AS monthly_growth_rate
FROM monthly_order_growth
WHERE year = 2025
ORDER BY order_loss_gain
"""

pd.read_sql(query,engine)

,year,month_name,order_count,previous_order_count,order_loss_gain,monthly_growth_rate
0,2025,February,1360,"1,623.00",-263.00,-16.20
1,2025,August,1518,"1,616.00",-98.00,-6.06
2,2025,October,1549,"1,623.00",-74.00,-4.56
3,2025,March,1543,"1,596.00",-53.00,-3.32
4,2025,July,1588,"1,640.00",-52.00,-3.17
5,2025,May,1593,"1,624.00",-31.00,-1.91
6,2025,June,1561,"1,577.00",-16.00,-1.01
7,2025,December,1545,"1,554.00",-9.00,-0.58
8,2025,November,1541,"1,531.00",10.00,0.65
9,2025,April,1567,"1,513.00",54.00,3.57


# Order Volume Insights (2025 vs. 2024)

- **Total order volume declined by 2.05% YoY (389 fewer orders), while AOV remained nearly unchanged (+0.07%), confirming that
lower sales volume—not pricing—was the primary driver of revenue decline.**
- **Networking products and category experienced the most severe decline**, making them the highest priority for product portfolio 
and demand generation initiatives.
- **Europe and South America** contributed the largest regional order losses, with **Brazil, Canada, UAE, and France** recording 
the biggest country-level declines.
- **SMB and Retail customers accounted for the majority of lost orders**, whereas the **Enterprise segment continued to grow**, 
indicating stronger resilience among larger organizations.
- **Finance and Healthcare industries experienced the greatest reductions in purchasing activity**, suggesting weaker demand across 
key commercial sectors.
- **February experienced the sharpest monthly decline (-16.20%)**, followed by weaker performance during **July–October**, 
indicating potential seasonal, operational, or market-related challenges.
- **Software products, Enterprise customers, and markets such as Saudi Arabia, Argentina, and the USA demonstrated positive growth**,
presenting opportunities for targeted investment and business expansion.

## Customer Churn Analysis

#### Customer Churn Count

In [32]:
query = """

SELECT COUNT(DISTINCT s.customer_id) AS churned_customer_count
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
WHERE d.year = 2024
AND NOT EXISTS 
(
    SELECT 1
    FROM fact_sales s2
    JOIN dim_date d2 ON s2.order_date = d2.date
    WHERE s2.customer_id = s.customer_id
    AND d2.year = 2025
)

"""

customer_churned_count = pd.read_sql(query,engine)
print(f"Churned Customer Count: {customer_churned_count.loc[0,'churned_customer_count']}")

Churned Customer Count: 114


#### Which customers stopped buying?

In [33]:
query = """
SELECT
    c.customer_id,
    c.customer_name,
    c.segment,
    c.industry,
    SUM(f.net_sales) AS revenue_2024,
    COUNT(DISTINCT f.order_id) AS orders_2024
FROM fact_sales f
JOIN dim_customer c
    ON f.customer_id = c.customer_id
JOIN dim_date d
    ON f.order_date = d.date
WHERE d.year = 2024
AND NOT EXISTS (
    SELECT 1
    FROM fact_sales f2
    JOIN dim_date d2
        ON f2.order_date = d2.date
    WHERE f2.customer_id = f.customer_id
      AND d2.year = 2025
)
GROUP BY
    c.customer_id,
    c.customer_name,
    c.segment,
    c.industry
ORDER BY revenue_2024 DESC

"""


churned_customer = pd.read_sql(query,engine)
display(churned_customer)

,customer_id,customer_name,segment,industry,revenue_2024,orders_2024
0,C02946,David Verma,Enterprise,Retail,"281,486.85",8
1,C04278,Meera Verma,Retail,IT,"186,437.80",5
2,C00269,Sneha Davis,Retail,Retail,"184,699.20",5
3,C02766,Divya Joshi,SMB,Finance,"177,177.50",5
4,C03425,Harish Singh,SMB,Finance,"172,916.00",8
...,...,...,...,...,...,...
109,C03628,Emily Kumar,Government,Healthcare,"6,580.00",1
110,C02948,Kavya Singh,Government,IT,"4,986.00",1
111,C04785,Olivia Kumar,Retail,IT,"3,336.00",1
112,C04408,Harish Iyer,Government,Retail,"2,534.40",1


#### Revenue Lost Due to Churn

In [34]:
query = """

SELECT SUM(s.net_sales) AS revenue_lost
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
WHERE d.year = 2024
AND NOT EXISTS 
(
    SELECT 1
    FROM fact_sales s2
    JOIN dim_date d2 ON s2.order_date = d2.date
    WHERE s2.customer_id = s.customer_id
    AND d2.year = 2025
)

"""

revenue_lost = pd.read_sql(query,engine)
print(f"Revenue Lost: {format_dollars(revenue_lost.loc[0,'revenue_lost'])}")

Revenue Lost: 9.35M


#### Top Lost Customers

In [35]:
query = """
SELECT c.customer_id,
       c.customer_name,
       SUM(s.net_sales) AS revenue
FROM fact_sales s
JOIN dim_customer c ON s.customer_id = c.customer_id
JOIN dim_date d ON s.order_date = d.date
WHERE d.year = 2024  
AND NOT EXISTS 
(
    SELECT 1
    FROM fact_sales s2
    JOIN dim_date d2 ON s2.order_date = d2.date
    WHERE s2.customer_id = s.customer_id 
    AND d2.year = 2025
)
GROUP BY c.customer_id,c.customer_name
ORDER BY revenue DESC
LIMIT 10
"""

top_lost_customers = pd.read_sql(query,engine)
display(top_lost_customers)

,customer_id,customer_name,revenue
0,C02946,David Verma,"281,486.85"
1,C04278,Meera Verma,"186,437.80"
2,C00269,Sneha Davis,"184,699.20"
3,C02766,Divya Joshi,"177,177.50"
4,C03425,Harish Singh,"172,916.00"
5,C04709,Ananya Smith,"172,028.70"
6,C00473,Daniel Davis,"169,418.85"
7,C01082,Liam Patel,"168,618.50"
8,C02735,Olivia Miller,"165,039.20"
9,C04196,Sneha Miller,"161,820.25"


#### Churn by Customer Segment

In [36]:
query = """

SELECT c.segment,
       COUNT(DISTINCT s.customer_id) AS churned_customer_count
FROM fact_sales s
JOIN dim_customer c ON s.customer_id = c.customer_id
JOIN dim_date d ON s.order_date = d.date
WHERE d.year = 2024
AND NOT EXISTS 
(
    SELECT 1
    FROM fact_sales s2
    JOIN dim_date d2 ON s2.order_date = d2.date
    WHERE s2.customer_id = s.customer_id
    AND d2.year = 2025
)
GROUP BY c.segment
"""

churn_by_segment = pd.read_sql(query,engine)
display(churn_by_segment)

,segment,churned_customer_count
0,Enterprise,26
1,Government,28
2,Retail,37
3,SMB,23


#### Churn by Industry

In [37]:
query = """

SELECT c.industry,
       COUNT(DISTINCT s.customer_id) AS churned_customer_count
FROM fact_sales s
JOIN dim_customer c ON s.customer_id = c.customer_id
JOIN dim_date d ON s.order_date = d.date
WHERE d.year = 2024
AND NOT EXISTS 
(
    SELECT 1
    FROM fact_sales s2
    JOIN dim_date d2 ON s2.order_date = d2.date
    WHERE s2.customer_id = s.customer_id
    AND d2.year = 2025
)
GROUP BY c.industry

"""

churn_by_industry = pd.read_sql(query,engine)
display(churn_by_industry)

,industry,churned_customer_count
0,Finance,27
1,Healthcare,28
2,IT,24
3,Retail,35


#### Churn by Region

In [38]:
query = """

SELECT
    g.region,
    COUNT(DISTINCT s.customer_id) AS churned_customer_count
FROM fact_sales s
JOIN dim_geography g
    ON s.geo_id = g.geo_id
JOIN dim_date d
    ON s.order_date = d.date
WHERE d.year = 2024
  AND NOT EXISTS (
      SELECT 1
      FROM fact_sales s2
      JOIN dim_date d2
          ON s2.order_date = d2.date
      WHERE s2.customer_id = s.customer_id
        AND d2.year = 2025
  )
GROUP BY g.region
ORDER BY churned_customer_count DESC;

"""

churn_by_region = pd.read_sql(query,engine)
display(churn_by_region)

,region,churned_customer_count
0,South America,72
1,North America,64
2,Europe,58
3,Middle East,58
4,Asia-Pacific,53


#### Churn By Country

In [39]:
query = """

SELECT
    g.country,
    COUNT(DISTINCT s.customer_id) AS churned_customer_count
FROM fact_sales s
JOIN dim_geography g
    ON s.geo_id = g.geo_id
JOIN dim_date d
    ON s.order_date = d.date
WHERE d.year = 2024
  AND NOT EXISTS (
      SELECT 1
      FROM fact_sales s2
      JOIN dim_date d2
          ON s2.order_date = d2.date
      WHERE s2.customer_id = s.customer_id
        AND d2.year = 2025
  )
GROUP BY g.country
ORDER BY churned_customer_count DESC;

"""

churn_by_country = pd.read_sql(query,engine)
display(churn_by_country)

,country,churned_customer_count
0,USA,50
1,Argentina,45
2,Brazil,44
3,Saudi Arabia,36
4,UAE,34
5,India,33
6,France,31
7,Canada,28
8,Germany,23
9,UK,18


#### Churn Rate

In [40]:
query = """
WITH customers_2024 AS
(
SELECT distinct(s.customer_id)
FROM fact_sales s
JOIN dim_date d ON s.order_date = d.date
WHERE d.year = 2024
),
churned AS 
(
SELECT customer_id
FROM customers_2024 c
WHERE NOT EXISTS 
(
    SELECT 1
    FROM fact_sales f
    JOIN dim_date d ON f.order_date = d.date
    WHERE f.customer_id = c.customer_id
    AND d.year = 2025
)
)
SELECT
      (SELECT COUNT(*) FROM churned) AS churned_customers,
      (SELECT COUNT(*) FROM customers_2024) AS customers_2024,
      ROUND(
        (SELECT COUNT(*) FROM churned) * 100.0 /
        (SELECT COUNT(*) FROM customers_2024),
        2
    ) AS churn_rate;
"""

pd.read_sql(query,engine)

,churned_customers,customers_2024,churn_rate
0,114,4885,2.33


# Churn Insights

- **Customer churn rate reached 2.33%, resulting in 114 lost customers and an estimated revenue loss of $9.35M.**
- **South America and North America experienced the highest customer attrition**, while the **USA, Argentina, and Brazil** 
recorded the largest number of churned customers.
- **Retail customers and the Retail industry experienced the highest churn**, indicating a need for targeted customer retention and
loyalty initiatives.
- Customer churn was distributed across all regions and industries, suggesting a **company-wide retention challenge rather than 
an isolated market issue.**
- A relatively small number of high-value customers accounted for a substantial portion of lost revenue, demonstrating that **improving retention
among strategic accounts could deliver a significant financial impact.**
- Implementing **early churn detection, customer health scoring, proactive account management, and loyalty programs** could help reduce
future churn and protect recurring revenue.